## Imports

In [61]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
import shap
import pandas as pd
from sklearn.model_selection import train_test_split

In [62]:
# Load data
diabetes = pd.read_csv("data/diabetes/diabetic_data.csv")
ids = pd.read_csv("data/diabetes/IDS_mapping.csv")

## Preprocessing

In [63]:
diabetes = diabetes.replace({'?': np.nan})
diabetes['readmitted'] = diabetes['readmitted'].replace({'NO': 0,'>30': 0,'<30': 1}).astype(int)
diabetes['diabetesMed'] = diabetes['diabetesMed'].replace({'No': 0, 'Yes': 1})
diabetes['change'] = diabetes['change'].replace({'No': 0, 'Ch': 1})
diabetes = diabetes.drop(columns=['encounter_id', 'patient_nbr'])
# Drop high-missing, messy columns
diabetes = diabetes.drop(columns=['weight', 'payer_code', 'medical_specialty'])
# Deleting rows with invalid gender in EDA we found there are 3
diabetes = diabetes[diabetes["gender"] != "Unknown/Invalid"]
# Replace missing race values with Unknown
diabetes["race"] = diabetes["race"].fillna("Unknown")
# Encoding age groups as numeric values 
age_map = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}

diabetes['age'] = diabetes['age'].map(age_map)

### Groups of interest
sex = diabetes["gender"].values
races = diabetes["race"].values
print(np.unique(sex))
print(np.unique(races))


['Female' 'Male']
['AfricanAmerican' 'Asian' 'Caucasian' 'Hispanic' 'Other' 'Unknown']


In [64]:
diabetes['metformin'].unique()

<StringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str

In [65]:
medication_cols = [
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'acetohexamide',
    'glipizide',
    'glyburide',
    'tolbutamide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'examide',
    'citoglipton',
    'insulin',
    'glyburide-metformin',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

for col in medication_cols:
    diabetes[col] = diabetes[col].apply(
        lambda x: 0 if x == 'No' else 1
    )

In [66]:
medication_counts = diabetes[medication_cols].sum().sort_values(ascending=False)

print(medication_counts)

insulin                     54383
metformin                   19987
glipizide                   12685
glyburide                   10650
pioglitazone                 7327
rosiglitazone                6364
glimepiride                  5191
repaglinide                  1539
glyburide-metformin           706
nateglinide                   703
acarbose                      308
chlorpropamide                 86
tolazamide                     39
miglitol                       38
tolbutamide                    23
glipizide-metformin            13
troglitazone                    3
metformin-rosiglitazone         2
acetohexamide                   1
glimepiride-pioglitazone        1
metformin-pioglitazone          1
examide                         0
citoglipton                     0
dtype: int64


In [67]:
rare_meds = [
    'acarbose',
    'chlorpropamide',
    'tolazamide',
    'miglitol',
    'tolbutamide',
    'glipizide-metformin',
    'troglitazone',
    'metformin-rosiglitazone',
    'acetohexamide',
    'glimepiride-pioglitazone',
    'metformin-pioglitazone',
    'examide',
    'citoglipton'
]

diabetes["other_meds"] = (
    diabetes[rare_meds].sum(axis=1) > 0
).astype(int)

diabetes = diabetes.drop(columns=rare_meds)

In [68]:
diabetes['diag_1'].unique()

<StringArray>
['250.83',    '276',    '648',      '8',    '197',    '414',    '428',
    '398',    '434',  '250.7',
 ...
     '58',    '649',    '832',    '133',    '975',    '833',    '391',
    '690',     '10',    'V51']
Length: 717, dtype: str

In [69]:
def categorize_diagnosis(diag):
    if pd.isna(diag):
        return "Unknown"

    diag = str(diag)

    # V-codes
    if diag.startswith("V"):
        return "Supplementary"

    # E-codes
    if diag.startswith("E"):
        return "External Causes"

    try:
        diag_num = float(diag)

        # 001–139.9
        if 1 <= diag_num < 140:
            return "Infectious and Parasitic Diseases"

        # 140–239.9
        elif 140 <= diag_num < 240:
            return "Neoplasms"

        # 240–279.9
        elif 240 <= diag_num < 280:
            return "Endocrine, Nutritional and Metabolic Diseases"

        # 280–289.9
        elif 280 <= diag_num < 290:
            return "Blood and Blood-Forming Diseases"

        # 290–319
        elif 290 <= diag_num < 320:
            return "Mental Disorders"

        # 320–389.9
        elif 320 <= diag_num < 390:
            return "Nervous System and Sense Organs"

        # 390–459.9
        elif 390 <= diag_num < 460:
            return "Circulatory System Diseases"

        # 460–519.9
        elif 460 <= diag_num < 520:
            return "Respiratory System Diseases"

        # 520–579.9
        elif 520 <= diag_num < 580:
            return "Digestive System Diseases"

        # 580–629.9
        elif 580 <= diag_num < 630:
            return "Genitourinary System Diseases"

        # 630–676.9
        elif 630 <= diag_num < 677:
            return "Pregnancy and Childbirth Complications"

        # 680–709.9
        elif 680 <= diag_num < 710:
            return "Skin and Subcutaneous Tissue Diseases"

        # 710–739.9
        elif 710 <= diag_num < 740:
            return "Musculoskeletal and Connective Tissue Diseases"

        # 740–759.9
        elif 740 <= diag_num < 760:
            return "Congenital Anomalies"

        # 760–779.9
        elif 760 <= diag_num < 780:
            return "Perinatal Conditions"

        # 780–799.9
        elif 780 <= diag_num < 800:
            return "Symptoms, Signs and Ill-Defined Conditions"

        # 800–999.9
        elif 800 <= diag_num < 1000:
            return "Injury and Poisoning"

        else:
            return "Other"

    except:
        return "Other"

In [70]:
diabetes["diag_1_group"] = diabetes["diag_1"].apply(categorize_diagnosis)
diabetes["diag_2_group"] = diabetes["diag_2"].apply(categorize_diagnosis)
diabetes["diag_3_group"] = diabetes["diag_3"].apply(categorize_diagnosis)
diabetes["diag_1_group"].value_counts()

diag_1_group
Circulatory System Diseases                       30335
Endocrine, Nutritional and Metabolic Diseases     11459
Respiratory System Diseases                       10407
Digestive System Diseases                          9208
Symptoms, Signs and Ill-Defined Conditions         7636
Injury and Poisoning                               6972
Genitourinary System Diseases                      5078
Musculoskeletal and Connective Tissue Diseases     4957
Neoplasms                                          3433
Infectious and Parasitic Diseases                  2768
Skin and Subcutaneous Tissue Diseases              2530
Mental Disorders                                   2262
Supplementary                                      1644
Nervous System and Sense Organs                    1211
Blood and Blood-Forming Diseases                   1103
Pregnancy and Childbirth Complications              687
Congenital Anomalies                                 51
Unknown                            

In [71]:
# List all diagnosis group columns
diag_group_cols = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
]

# Get all unique disease categories
all_categories = pd.unique(
    diabetes[diag_group_cols].values.ravel()
)

# Create binary columns
for category in all_categories:
    diabetes[category] = (
        (diabetes["diag_1_group"] == category) |
        (diabetes["diag_2_group"] == category) |
        (diabetes["diag_3_group"] == category)
    ).astype(int)

In [72]:
diabetes = diabetes.drop(columns=[
    "diag_1",
    "diag_2",
    "diag_3",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
])

In [73]:
target_name = "readmitted"

In [74]:
# Find all categorical/string columns automatically
categorical_cols = diabetes.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(categorical_cols)

['race', 'gender', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']


In [75]:
diabetes['gender'] = diabetes['gender'].replace({'Male': 1, 'Female': 0})

categorical_cols = diabetes.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()


# One-hot encode categorical variables
diabetes_encoded = pd.get_dummies(
    diabetes,
    columns=categorical_cols,
    drop_first=True,
    dummy_na=False
)

print(diabetes_encoded.shape)
diabetes_encoded.head()

(101763, 55)


,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,race_Hispanic,race_Other,race_Unknown,gender_1,max_glu_serum_>300,max_glu_serum_Norm,A1Cresult_>8,A1Cresult_Norm,change_1,diabetesMed_1
0,5,6,25,1,1,41,0,1,0,0,...,False,False,False,False,False,False,False,False,False,False
1,15,1,1,7,3,59,0,18,0,0,...,False,False,False,False,False,False,False,False,True,True
2,25,1,1,7,2,11,5,13,2,0,...,False,False,False,False,False,False,False,False,False,True
3,35,1,1,7,2,44,1,16,0,0,...,False,False,False,True,False,False,False,False,True,True
4,45,1,1,7,1,51,0,8,0,0,...,False,False,False,True,False,False,False,False,True,True


In [76]:
X = diabetes_encoded[[c for c in diabetes_encoded if c != target_name]].copy()
y = diabetes_encoded[target_name].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

In [77]:
def print_classification_results(y_test, y_pred):
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print('Classification Report:')
    print(classification_report(y_test, y_pred))

In [78]:
def get_log_reg_pipeline(
    seed: int = 70766,
    max_iter: int = 5000,
    penalty: str = 'l2',
    C: float = 0.8497534359086438,
    tol: float = 1e-4,
    solver: str = 'saga'
):
    scaler = StandardScaler() # Standard scale for log reg
    model = LogisticRegression(
        class_weight='balanced',
        penalty = penalty,
        C = C,
        random_state = seed,
        max_iter = max_iter,
        tol = tol,
        solver = solver
    )
    pipeline = Pipeline([('scaler', scaler), ('classifier', model)])
    return pipeline

In [79]:
# Get pipeline and fit to training data
log_reg = get_log_reg_pipeline()
log_reg.fit(X_train, y_train)

# Predict on test set
log_reg_y_pred = log_reg.predict(X_test)

# Evaluate
print_classification_results(y_test, log_reg_y_pred)

c:\Users\kubic\OneDrive\Dokumenty\envs\algotithmic_fairness\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.6673
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.69      0.79     18082
           1       0.17      0.51      0.25      2271

    accuracy                           0.67     20353
   macro avg       0.54      0.60      0.52     20353
weighted avg       0.83      0.67      0.73     20353



In [80]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, log_reg_y_pred))

[[12435  5647]
 [ 1124  1147]]
